In [7]:
#Data

input_data = [
    ("WarehouseA", "C1", "2024-03-15 09:00:00", "2024-03-15 11:00:00", "2024-03-15 09:30:00", "2024-03-15 12:00:00", "FedEx", "Weather"),
    ("WarehouseB", "C2", "2024-03-15 12:00:00", "2024-03-15 15:00:00", "2024-03-15 15:30:00", "2024-03-15 18:40:00", "UPS", "Operational"),
    ("WarehouseC", "C3", "2024-03-16 09:50:00", "2024-03-16 13:30:00", "2024-03-16 10:20:00", "2024-03-16 13:50:00", "DHL", "Traffic"),
    ("WarehouseD", "C2", "2024-03-16 15:00:00", "2024-03-16 20:00:00", "2024-03-16 15:30:00", "2024-03-16 20:50:00", "Amazon Logistics", "Technical"),
    ("WarehouseE", "C4", "2024-03-17 08:30:00", "2024-03-17 10:30:00", "2024-03-17 09:00:00", "2024-03-17 11:10:00", "FedEx", "Weather"),
    ("WarehouseF", "C5", "2024-03-17 11:00:00", "2024-03-17 14:00:00", "2024-03-17 11:15:00", "2024-03-17 14:30:00", "UPS", "Operational"),
    ("WarehouseA", "C6", "2024-03-18 07:45:00", "2024-03-18 12:15:00", "2024-03-18 08:00:00", "2024-03-18 12:40:00", "DHL", "Traffic"),
    ("WarehouseB", "C4", "2024-03-18 14:00:00", "2024-03-18 18:00:00", "2024-03-18 14:30:00", "2024-03-18 19:00:00", "Amazon Logistics", "Technical"),
    ("WarehouseC", "C6", "2024-03-19 09:00:00", "2024-03-19 13:00:00", "2024-03-19 09:20:00", "2024-03-19 13:20:00", "FedEx", "Weather"),
    ("WarehouseD", "C7", "2024-03-19 10:30:00", "2024-03-19 15:30:00", "2024-03-19 10:45:00", "2024-03-19 16:00:00", "UPS", "Operational"),
    ("WarehouseE", "C8", "2024-03-20 08:00:00", "2024-03-20 12:00:00", "2024-03-20 08:15:00", "2024-03-20 12:15:00", "DHL", "Traffic"),
    ("WarehouseF", "C2", "2024-03-20 13:00:00", "2024-03-20 17:00:00", "2024-03-20 13:30:00", "2024-03-20 18:00:00", "Amazon Logistics", "Technical"),
    ("WarehouseG", "C9", "2024-03-21 11:30:00", "2024-03-21 14:30:00", "2024-03-21 12:00:00", "2024-03-21 15:00:00", "FedEx", "Weather"),
    ("WarehouseA", "C10", "2024-03-21 09:15:00", "2024-03-21 13:15:00", "2024-03-21 09:45:00", "2024-03-21 13:50:00", "UPS", "Operational"),
]

col_format = [
    "origin_warehouse", "customer", "scheduled_departure_time",
    "scheduled_arrival_time", "actual_departure_time", "actual_arrival_time", "carrier", "delay_reason"]

In [1]:
#Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [4]:
#Create SparkSession
spark = SparkSession.builder.getOrCreate()

In [28]:
#Create Dataframe using input_data & col_format
df = spark.createDataFrame(input_data,col_format)
# df=df.withColumn("scheduled_departure_time", F.unix_timestamp("scheduled_departure_time"))
# df=df.withColumn("scheduled_arrival_time", F.unix_timestamp("scheduled_arrival_time"))
# df=df.withColumn("actual_departure_time", F.unix_timestamp("actual_departure_time"))
# df=df.withColumn("actual_arrival_time", F.unix_timestamp("actual_arrival_time"))
df.printSchema()

root
 |-- origin_warehouse: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- scheduled_departure_time: string (nullable = true)
 |-- scheduled_arrival_time: string (nullable = true)
 |-- actual_departure_time: string (nullable = true)
 |-- actual_arrival_time: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- delay_reason: string (nullable = true)



In [35]:
#Filter the dataframe for delay greater than 20 minutes
df=df.withColumn("delay", (F.unix_timestamp("actual_departure_time")-F.unix_timestamp("scheduled_departure_time")))
df.filter((F.col("delay")/60) > 20).show()

+----------------+--------+------------------------+----------------------+---------------------+-------------------+----------------+------------+-----+
|origin_warehouse|customer|scheduled_departure_time|scheduled_arrival_time|actual_departure_time|actual_arrival_time|         carrier|delay_reason|delay|
+----------------+--------+------------------------+----------------------+---------------------+-------------------+----------------+------------+-----+
|      WarehouseA|      C1|     2024-03-15 09:00:00|   2024-03-15 11:00:00|  2024-03-15 09:30:00|2024-03-15 12:00:00|           FedEx|     Weather| 1800|
|      WarehouseB|      C2|     2024-03-15 12:00:00|   2024-03-15 15:00:00|  2024-03-15 15:30:00|2024-03-15 18:40:00|             UPS| Operational|12600|
|      WarehouseC|      C3|     2024-03-16 09:50:00|   2024-03-16 13:30:00|  2024-03-16 10:20:00|2024-03-16 13:50:00|             DHL|     Traffic| 1800|
|      WarehouseD|      C2|     2024-03-16 15:00:00|   2024-03-16 20:00:00| 

In [36]:
#Calculate average delay per customer
df.groupby("customer").agg(F.avg("delay")).show()

[Stage 24:>                                                         (0 + 2) / 2]

+--------+----------+
|customer|avg(delay)|
+--------+----------+
|      C6|    1050.0|
|      C3|    1800.0|
|      C4|    1800.0|
|      C5|     900.0|
|      C1|    1800.0|
|      C2|    5400.0|
|      C9|    1800.0|
|      C8|     900.0|
|      C7|     900.0|
|     C10|    1800.0|
+--------+----------+



In [37]:
#Calculate average delay per carrier
df.groupby("carrier").agg(F.avg("delay")).show()

+----------------+----------+
|         carrier|avg(delay)|
+----------------+----------+
|             DHL|    1200.0|
|           FedEx|    1650.0|
|Amazon Logistics|    1800.0|
|             UPS|    4050.0|
+----------------+----------+



In [38]:
#Calculate average delay per reason
df.groupby("delay_reason").agg(F.avg("delay")).show()

+------------+----------+
|delay_reason|avg(delay)|
+------------+----------+
|     Weather|    1650.0|
| Operational|    4050.0|
|     Traffic|    1200.0|
|   Technical|    1800.0|
+------------+----------+



In [39]:
#Calculate average delay per Warehouse
df.groupby("origin_warehouse").agg(F.avg("delay")).show()

+----------------+----------+
|origin_warehouse|avg(delay)|
+----------------+----------+
|      WarehouseA|    1500.0|
|      WarehouseD|    1350.0|
|      WarehouseF|    1350.0|
|      WarehouseC|    1500.0|
|      WarehouseE|    1350.0|
|      WarehouseB|    7200.0|
|      WarehouseG|    1800.0|
+----------------+----------+



In [42]:
#Identify frequently delayed routes (Output dataframe should have two columns, delay reason & avg delay)
df=df.withColumn("routes",F.concat(F.col("origin_warehouse"),F.lit("-"),F.col("customer")))
df.groupby("routes").count().orderBy(F.col("count").desc()).select("routes","count","delay_reason")

+--------------+-----+
|        routes|count|
+--------------+-----+
| WarehouseD-C2|    1|
| WarehouseA-C6|    1|
| WarehouseE-C4|    1|
| WarehouseA-C1|    1|
| WarehouseF-C5|    1|
| WarehouseB-C2|    1|
| WarehouseC-C3|    1|
| WarehouseE-C8|    1|
| WarehouseC-C6|    1|
| WarehouseD-C7|    1|
|WarehouseA-C10|    1|
| WarehouseG-C9|    1|
| WarehouseF-C2|    1|
| WarehouseB-C4|    1|
+--------------+-----+

